# 04. Feature Engineering & Graph Construction

Αυτό το Notebook αποτελεί τον «συνδετικό κρίκο» μεταξύ της **Εξερευνητικής Ανάλυσης (EDA)** και της **Φάσης Εκπαίδευσης των Μοντέλων**. Στόχος είναι η μετατροπή των ακατέργαστων δεδομένων (3.3M εγγραφές) σε μια πολυδιάστατη δομή χαρακτηριστικών που θα υποστηρίξει τόσο τις παραδοσιακές όσο και τις state-of-the-art αρχιτεκτονικές (Mamba, GNN, Transformers).

---

##  Αντικειμενικοί Στόχοι

1. **Temporal Encoding:** Μετατροπή των χρονικών δεδομένων σε κυκλικές μεταβλητές ($sin/cos$) για την αποτύπωση της περιοδικότητας.
2. **Lag Engineering:** Δημιουργία χρονικών υστερήσεων ($t-n$) για την υποστήριξη των Selective State Spaces του **Mamba**.
3. **Statistical Aggregations:** Υπολογισμός κινητών μέσων όρων (Rolling Windows) για τις **5 παραδοσιακές μεθόδους πρόβλεψης**.
4. **Graph Construction:** Δημιουργία του Πίνακα Γειτνίασης (Adjacency Matrix) βάσει γεωγραφικών συντεταγμένων (Lat/Lon) για την υποστήριξη **Graph Neural Networks (GNN)**.

---

##  Σχεδιασμός Χαρακτηριστικών ανά Αρχιτεκτονική

| Αρχιτεκτονική | Τύπος Χαρακτηριστικών | Σκοπός |
| :--- | :--- | :--- |
| **GNN** | Spatial Adjacency Matrix & Distance Weights | Μοντελοποίηση χωρικής συσχέτισης ανέμου. |
| **Mamba** | State Transitions & Long-range Lags | Βελτιστοποίηση Linear-time Sequence Modeling. |
| **Transformers** | Cyclical Positional Encodings | Αποτύπωση εποχικότητας και χρονικής προσοχής. |
| **Traditional (5)** | Aggregate Stats (Mean, Std, Min/Max) | Δημιουργία ισχυρών Baseline μοντέλων. |

---

##  Βιβλιοθήκες & Εργαλεία
* `numpy` & `pandas`: Επεξεργασία δεδομένων μεγάλης κλίμακας.
* `scipy.spatial`: Υπολογισμός Ευκλείδειας απόστασης για τον γράφο.
* `sklearn.preprocessing`: Κανονικοποίηση (Scaling) χαρακτηριστικών.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler
from dotenv import load_dotenv

# Φόρτωση ρυθμίσεων από το .env (για το API Token αν χρειαστεί)
load_dotenv()

# Ρυθμίσεις εμφάνισης
%matplotlib inline
pd.set_option('display.max_columns', None)
print("Βιβλιοθήκες φορτώθηκαν επιτυχώς.")

Βιβλιοθήκες φορτώθηκαν επιτυχώς.


In [6]:
# Cell 2: Εισαγωγή και Διόρθωση Μικτών Formats Ημερομηνίας
DATA_PATH = '../data/processed/master_dataset.csv'

try:
    df_master = pd.read_csv(DATA_PATH)
    
    # Χρησιμοποιούμε format='mixed' για να χειριστούμε ταυτόχρονα "13/12/2018" και "2018-12-11"
    df_master['fcst_time'] = pd.to_datetime(df_master['fcst_time'], format='mixed', dayfirst=True)
    
    # Μετονομασία σε 'timestamp' για ομοιομορφία στον κώδικα
    df_master = df_master.rename(columns={'fcst_time': 'timestamp'})
    
    # Ταξινόμηση (κρίσιμο για Mamba και χρονοσειρές)
    df_master = df_master.sort_values('timestamp').reset_index(drop=True)
    
    print(f"Επιτυχία! Το Master Dataset φορτώθηκε και καθαρίστηκε.")
    print(f"Σχήμα: {df_master.shape}")
    print(f"Εύρος: {df_master['timestamp'].min()} έως {df_master['timestamp'].max()}")
    display(df_master.head())
    
except Exception as e:
    print(f"Σφάλμα: {e}")

Επιτυχία! Το Master Dataset φορτώθηκε και καθαρίστηκε.
Σχήμα: (1658860, 25)
Εύρος: 2018-01-25 00:00:00 έως 2020-12-05 23:00:00


,timestamp,nwp_fcst_horiz_hours,T_HAG_2_M,RELHUM_HAG_2_M,PS_SFC_0_M,U_GVL_58_HL,V_GVL_58_HL,U_GVL_60_HL,V_GVL_60_HL,ASWDIFDS_SFC_0_M,ASWDIRS_SFC_0_M,U_GVL_58_HL_m1,V_GVL_58_HL_m1,U_GVL_60_HL_m1,V_GVL_60_HL_m1,U_GVL_58_HL_p1,V_GVL_58_HL_p1,U_GVL_60_HL_p1,V_GVL_60_HL_p1,ws_ref,Wind_Speed_100m_ms,test_flag,Power_Output_Normalized,Baseline_Prediction,park_id
0,2018-01-25 00:00:00,24,274.281,89.110,95417.363,-0.019,1.691,0.226,0.085,22.059,58.965,-0.296,2.153,0.349,0.512,-0.032,1.763,0.418,0.611,0.001691,0.001828,0,0.001,0.001,7308
1,2018-01-25 01:00:00,25,274.224,88.113,95397.996,-0.296,2.153,0.349,0.512,21.177,56.605,0.109,1.605,0.509,-0.036,-0.019,1.691,0.226,0.085,0.002173,0.002349,0,0.000,0.002,7308
2,2018-01-25 02:00:00,26,274.131,87.148,95366.211,0.109,1.605,0.509,-0.036,20.362,54.430,0.214,1.137,0.484,0.283,-0.296,2.153,0.349,0.512,0.001609,0.001739,0,0.001,0.000,7308
3,2018-01-25 03:00:00,27,273.720,90.329,95324.062,0.214,1.137,0.484,0.283,19.608,52.414,-0.223,2.010,0.282,0.580,0.109,1.605,0.509,-0.036,0.001157,0.001251,0,0.000,0.000,7308
4,2018-01-25 04:00:00,28,273.705,91.284,95330.480,-0.223,2.010,0.282,0.580,18.908,50.543,-0.571,2.487,0.042,0.618,0.214,1.137,0.484,0.283,0.002022,0.002186,0,0.000,0.001,7308


In [7]:
# Cell 3: Cyclical Temporal Encoding
def encode_cyclical(df, col, max_val):
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

# Εξαγωγή χαρακτηριστικών
df_master['hour'] = df_master['timestamp'].dt.hour
df_master['month'] = df_master['timestamp'].dt.month

# Εφαρμογή κωδικοποίησης
df_master = encode_cyclical(df_master, 'hour', 24)
df_master = encode_cyclical(df_master, 'month', 12)

print("Η κυκλική κωδικοποίηση ολοκληρώθηκε!")
display(df_master[['timestamp', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']].head())

Η κυκλική κωδικοποίηση ολοκληρώθηκε!


,timestamp,hour_sin,hour_cos,month_sin,month_cos
0,2018-01-25 00:00:00,0.000000,1.000000,0.5,0.866025
1,2018-01-25 01:00:00,0.258819,0.965926,0.5,0.866025
2,2018-01-25 02:00:00,0.500000,0.866025,0.5,0.866025
3,2018-01-25 03:00:00,0.707107,0.707107,0.5,0.866025
4,2018-01-25 04:00:00,0.866025,0.500000,0.5,0.866025


In [8]:
# Cell 4: Lag Engineering (Δημιουργία "Μνήμης" για το μοντέλο)
# Ορίζουμε τις στήλες που θέλουμε να έχουν lags (κυρίως την ισχύ και τον άνεμο)
features_to_lag = ['Power_Output_Normalized', 'Wind_Speed_100m_ms']
lags = [1, 3, 6] # Υστερήσεις σε ώρες

for col in features_to_lag:
    for lag in lags:
        # Δημιουργούμε το lag ανά πάρκο (park_id) για να μην μπερδευτούν τα δεδομένα
        df_master[f'{col}_lag_{lag}'] = df_master.groupby('park_id')[col].shift(lag)

# Αφαιρούμε τις γραμμές με NaN που δημιουργήθηκαν από τα shifts
df_master = df_master.dropna().reset_index(drop=True)

print(f"Το Lag Engineering ολοκληρώθηκε για {lags} ώρες.")
print(f"Νέο σχήμα δεδομένων (μετά το dropna): {df_master.shape}")
display(df_master.head())

Το Lag Engineering ολοκληρώθηκε για [1, 3, 6] ώρες.
Νέο σχήμα δεδομένων (μετά το dropna): (1657234, 37)


,timestamp,nwp_fcst_horiz_hours,T_HAG_2_M,RELHUM_HAG_2_M,PS_SFC_0_M,U_GVL_58_HL,V_GVL_58_HL,U_GVL_60_HL,V_GVL_60_HL,ASWDIFDS_SFC_0_M,ASWDIRS_SFC_0_M,U_GVL_58_HL_m1,V_GVL_58_HL_m1,U_GVL_60_HL_m1,V_GVL_60_HL_m1,U_GVL_58_HL_p1,V_GVL_58_HL_p1,U_GVL_60_HL_p1,V_GVL_60_HL_p1,ws_ref,Wind_Speed_100m_ms,test_flag,Power_Output_Normalized,Baseline_Prediction,park_id,hour,month,hour_sin,hour_cos,month_sin,month_cos,Power_Output_Normalized_lag_1,Power_Output_Normalized_lag_3,Power_Output_Normalized_lag_6,Wind_Speed_100m_ms_lag_1,Wind_Speed_100m_ms_lag_3,Wind_Speed_100m_ms_lag_6
0,2018-01-25 06:00:00,30,274.538,84.870,95293.973,-0.675,2.661,0.162,0.486,17.648,47.172,-0.657,2.329,0.240,0.349,-0.571,2.487,0.042,0.618,0.002745,0.002968,0,0.0,0.007,7308,6,1,1.000000,6.123234e-17,0.500000,0.866025,0.000,0.000,0.001,0.002758,0.001251,0.001828
1,2018-01-25 07:00:00,31,274.764,81.978,95254.418,-0.657,2.329,0.240,0.349,17.083,45.652,-0.558,2.187,0.326,0.399,-0.675,2.661,0.162,0.486,0.002420,0.002616,0,0.0,0.004,7308,7,1,0.965926,-2.588190e-01,0.500000,0.866025,0.000,0.000,0.000,0.002968,0.002186,0.002349
2,2018-01-25 08:00:00,32,275.017,81.603,95242.051,-0.558,2.187,0.326,0.399,17.766,44.406,-0.460,1.863,0.234,0.339,-0.657,2.329,0.240,0.349,0.002257,0.002440,0,0.0,0.002,7308,8,1,0.866025,-5.000000e-01,0.500000,0.866025,0.000,0.000,0.001,0.002616,0.002758,0.001739
3,2018-01-25 11:00:00,35,278.518,86.377,95111.832,-0.327,1.427,-0.359,0.233,27.170,49.027,-0.406,0.721,-0.310,-0.180,-0.536,1.754,-0.133,0.386,0.001464,0.001583,0,0.0,0.000,7308,11,1,0.258819,-9.659258e-01,0.500000,0.866025,0.000,0.000,0.000,0.002440,0.002968,0.001251
4,2018-08-12 07:00:00,31,272.995,70.567,89405.664,6.775,2.505,3.020,1.218,17.945,8.727,7.381,2.637,3.146,1.250,8.239,1.803,3.816,0.829,0.007223,0.007808,0,0.0,0.348,1550,7,8,0.965926,-2.588190e-01,-0.866025,-0.500000,0.001,0.026,0.005,0.009117,0.006621,0.010386


In [9]:
# Cell 5: Rolling Statistics (Κινητοί Μέσοι Όροι)
window_size = 6 # Παράθυρο 6 ωρών

# Υπολογισμός μέσου όρου και τυπικής απόκλισης
df_master['power_rolling_mean_6h'] = df_master.groupby('park_id')['Power_Output_Normalized'].transform(lambda x: x.rolling(window=window_size).mean())
df_master['power_rolling_std_6h'] = df_master.groupby('park_id')['Power_Output_Normalized'].transform(lambda x: x.rolling(window=window_size).std())

# Αφαίρεση NaN που προέκυψαν από το rolling window
df_master = df_master.dropna().reset_index(drop=True)

print("Οι κινητοί στατιστικοί δείκτες υπολογίστηκαν.")
display(df_master[['timestamp', 'power_rolling_mean_6h', 'power_rolling_std_6h']].head())

Οι κινητοί στατιστικοί δείκτες υπολογίστηκαν.


,timestamp,power_rolling_mean_6h,power_rolling_std_6h
0,2018-08-12 15:00:00,0.020667,0.015475
1,2018-08-12 16:00:00,0.021833,0.013776
2,2018-08-12 17:00:00,0.017167,0.011805
3,2018-08-12 18:00:00,0.016000,0.013161
4,2018-08-12 18:00:00,0.003667,0.006250
